# S41. Teste de Média — Unilateral à Esquerda
## Left-Tailed Test for Mean

[◀ Anterior](S40_Teste_Proporcao_Bicaudal.ipynb) | [📖 Índice](S00_Index_Estatistica.ipynb) | [Próximo ▶](S42_Teste_Media_Bicaudal.ipynb)

## 🎯 Objetivos de Aprendizagem

- Compreender o teste T unilateral à esquerda para média
- Formular H₀: μ ≥ μ₀ e H₁: μ < μ₀
- Identificar a região de rejeição à esquerda na distribuição t
- Implementar o left-tailed t-test manualmente e com scipy
- Calcular o p-valor unilateral a partir do ttest_1samp

## 📝 Introdução

O **teste T unilateral à esquerda** (left-tailed t-test) para média é usado quando queremos verificar se a **média populacional é menor** que um determinado valor. Como o desvio padrão populacional é geralmente desconhecido, usamos a distribuição t de Student.

Situações típicas:
- "O tempo de carregamento do site diminuiu após a otimização?"
- "A pressão arterial média dos pacientes reduziu com o tratamento?"
- "O consumo médio de combustível é menor que o anunciado pelo fabricante?"

Este teste é direcional: queremos detectar se a média é **menor** que μ₀.

## 📚 Explicação Detalhada

### 1. Formulação do Teste

- **H₀**: μ ≥ μ₀ (a média é maior ou igual ao valor de referência)
- **H₁**: μ < μ₀ (a média é menor que o valor de referência)

A região de rejeição está na **cauda esquerda** da distribuição t com gl = n - 1.

### 2. Estatística t e Valor Crítico

t = (x̄ - μ₀) / (s / √n)

Rejeitamos H₀ se:
- **t calculado < -t_{α, gl}** (região crítica)
- **p-valor < α** (o p-valor unilateral à esquerda)

O p-valor unilateral à esquerda = P(T ≤ t_calc) = t.cdf(t_calc, gl)

### 3. Obtendo o p-valor Unilateral do ttest_1samp

O `ttest_1samp` do scipy retorna o p-valor **bicaudal**. Para obter o p-valor unilateral à esquerda:

- Se t < 0: p_unilateral_esq = p_bicaudal / 2
- Se t > 0: p_unilateral_esq = 1 - p_bicaudal / 2

## 💻 Exemplos Práticos

In [ ]:
# Importando bibliotecas
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

print("Bibliotecas carregadas com sucesso!")

In [ ]:
# Exemplo 1: Teste T unilateral a esquerda (calculo manual)
# Um fabricante afirma que suas lampadas duram em media 1000 horas.
# Desconfia-se que duram menos. Teste com 12 lampadas.

lampadas = np.array([985, 992, 978, 1005, 988, 996,
                     980, 1002, 975, 991, 987, 994])
mu_0 = 1000.0
alpha = 0.05

# Calculo manual
n = len(lampadas)
media = np.mean(lampadas)
desvio = np.std(lampadas, ddof=1)
gl = n - 1
erro = desvio / np.sqrt(n)
t = (media - mu_0) / erro

# p-valor unilateral a esquerda
p_esquerda = stats.t.cdf(t, df=gl)

# Valor critico para teste unilateral esquerda
t_crit = stats.t.ppf(alpha, df=gl)

print(f"Lâmpadas: {lampadas}")
print(f"n = {n}, média = {media:.2f}, desvio = {desvio:.2f}")
print(f"H₀: μ ≥ {mu_0} | H₁: μ < {mu_0}")
print(f"t = {t:.4f} (gl = {gl})")
print(f"t crítico = {t_crit:.4f}")
print(f"p-valor (unilateral esq.) = {p_esquerda:.4f}")
print()
if t < t_crit:
    print(f"✅ t ({t:.4f}) < t crítico ({t_crit:.4f}) → Rejeitamos H₀")
    print("   Evidência de que as lâmpadas duram menos que 1000h.")
else:
    print(f"❌ t ({t:.4f}) ≥ t crítico ({t_crit:.4f}) → Não rejeitamos H₀")

In [ ]:
# Exemplo 2: Usando ttest_1samp e convertendo para unilateral
t_stat, p_bi = stats.ttest_1samp(lampadas, mu_0)

# Convertendo p-valor bicaudal para unilateral esquerda
if t_stat < 0:
    p_uni_esq = p_bi / 2
else:
    p_uni_esq = 1 - p_bi / 2

print("Usando scipy.stats.ttest_1samp:")
print(f"t = {t_stat:.4f}")
print(f"p-valor (bicaudal) = {p_bi:.4f}")
print(f"p-valor (unilateral esq.) = {p_uni_esq:.4f}")
if p_uni_esq < alpha:
    print(f"✅ p-valor {p_uni_esq:.4f} < α = {alpha} → Rejeitamos H₀")
else:
    print(f"❌ p-valor {p_uni_esq:.4f} ≥ α = {alpha} → Não rejeitamos H₀")

In [ ]:
# Exemplo 3: Visualizacao da regiao de rejeicao a esquerda (t)
x = np.linspace(-4, 4, 1000)
y = stats.t.pdf(x, df=gl)

plt.figure(figsize=(10, 6))
plt.plot(x, y, 'b-', label=f'Distribuição t (gl = {gl})')

# Regiao de rejeicao
x_fill = np.linspace(-4, t_crit, 100)
plt.fill_between(x_fill, stats.t.pdf(x_fill, df=gl), color='red', alpha=0.3, label='Região de Rejeição')

plt.axvline(t, color='green', linestyle='--', lw=2, label=f't = {t:.2f}')
plt.axvline(t_crit, color='red', linestyle=':', lw=2, label=f't crítico = {t_crit:.2f}')
plt.title('Teste T Unilateral à Esquerda')
plt.xlabel('t')
plt.ylabel('Densidade')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Exemplo 4: Funcao completa para teste t unilateral esquerda
def ttest_unilateral_esquerda(dados, mu_0, alpha=0.05):
    """Teste T unilateral a esquerda para media."""
    n = len(dados)
    media = np.mean(dados)
    desvio = np.std(dados, ddof=1)
    gl = n - 1
    t_stat = (media - mu_0) / (desvio / np.sqrt(n))
    p_esq = stats.t.cdf(t_stat, df=gl)
    t_crit = stats.t.ppf(alpha, df=gl)
    
    print(f"=== Teste T Unilateral à Esquerda ===")
    print(f"n = {n}, média = {media:.3f}, desvio = {desvio:.3f}")
    print(f"H₀: μ ≥ {mu_0} | H₁: μ < {mu_0}")
    print(f"t = {t_stat:.4f} (gl = {gl})")
    print(f"t crítico = {t_crit:.4f}")
    print(f"p-valor = {p_esq:.4f}")
    if t_stat < t_crit:
        print(f"✅ Rejeitamos H₀: μ < {mu_0}")
    else:
        print(f"❌ Não rejeitamos H₀")
    return {"t": t_stat, "p": p_esq, "gl": gl}

ttest_unilateral_esquerda(lampadas, 1000)
print()
ttest_unilateral_esquerda(np.array([23, 21, 22, 20, 24, 22, 21, 20]), 22.5)

In [ ]:
# Exemplo 5: Teste unilateral com dados gerados
# Simulando dados de um tratamento que reduziu o peso
np.random.seed(42)
perda_peso = np.random.normal(loc=3.5, scale=1.2, size=25)

t_stat, p_bi = stats.ttest_1samp(perda_peso, 4.0)
if t_stat < 0:
    p_esq = p_bi / 2
else:
    p_esq = 1 - p_bi / 2

print("Teste: A perda de peso média é menor que 4kg?")
print(f"Média amostral: {np.mean(perda_peso):.2f}")
print(f"t = {t_stat:.4f}")
print(f"p-valor (unilateral esq.) = {p_esq:.4f}")
if p_esq < 0.05:
    print("✅ Sim, a perda de peso é significativamente menor que 4kg.")
else:
    print("❌ Não há evidência suficiente.")

## ✅ Exercícios Resolvidos

**Exercício 1:** Um fabricante de baterias afirma que a duração média é de 500 horas. Um teste com 10 baterias mostra: [485, 492, 478, 505, 488, 496, 480, 502, 475, 491]. Teste se a duração é menor que 500h (α = 0.05).

In [ ]:
# Solucao do Exercicio 1
baterias = np.array([485, 492, 478, 505, 488, 496, 480, 502, 475, 491])
t_stat, p_bi = stats.ttest_1samp(baterias, 500)
p_esq = p_bi / 2 if t_stat < 0 else 1 - p_bi / 2

print(f"Média: {np.mean(baterias):.1f}")
print(f"t = {t_stat:.4f}, p (unilateral esq.) = {p_esq:.4f}")
if p_esq < 0.05:
    print("✅ Rejeitamos H₀: duração < 500h.")
else:
    print("❌ Não rejeitamos H₀.")

**Exercício 2:** Uma escola afirma que a nota média é ≥ 75. Amostra de 20 alunos: média 72.5, desvio 4.2. Teste a 1%.

In [ ]:
# Solucao do Exercicio 2 (dados agregados)
n = 20
media = 72.5
desvio = 4.2
mu_0 = 75.0
alpha = 0.01

t = (media - mu_0) / (desvio / np.sqrt(n))
p = stats.t.cdf(t, df=n-1)

print(f"H₀: μ ≥ {mu_0} | H₁: μ < {mu_0}")
print(f"t = {t:.4f}, p-valor = {p:.4f}")
if p < alpha:
    print(f"✅ Rejeitamos H₀ a {alpha}: a média é menor que {mu_0}.")
else:
    print(f"❌ Não rejeitamos H₀ a {alpha}.")

**Exercício 3:** Um programa de economia de energia afirma que o consumo médio é ≤ 250 kWh/mês. 30 casas apresentam média 242 kWh e desvio 18 kWh. Teste a 5%.

In [ ]:
# Solucao do Exercicio 3
n = 30
media = 242
desvio = 18
mu_0 = 250
alpha = 0.05

t = (media - mu_0) / (desvio / np.sqrt(n))
p = stats.t.cdf(t, df=n-1)

print(f"H₀: μ ≥ {mu_0} | H₁: μ < {mu_0}")
print(f"t = {t:.4f}, p-valor = {p:.4f}")
if p < alpha:
    print("✅ Rejeitamos H₀: consumo médio < 250 kWh.")
else:
    print("❌ Não rejeitamos H₀.")

## 📋 Exercícios Propostos

| Nível | Exercício |
|-------|-----------|
| 🟢 Fácil | 1. Teste se a média de [10, 9, 11, 8, 10, 9] é menor que 11 (α = 0.05). |
| 🟡 Médio | 2. Um fabricante afirma que o peso médio é ≥ 200g. 15 itens: média 197g, desvio 4g. Teste a 1%. |
| 🔴 Difícil | 3. Crie uma função que realize o teste T unilateral (esquerda ou direita) a partir dos dados brutos. |

In [ ]:
# 🟢 Exercício 1: Escreva sua solução aqui
dados = np.array([10, 9, 11, 8, 10, 9])
# Complete


In [ ]:
# 🟡 Exercício 2: Escreva sua solução aqui
n = 15
media = 197
desvio = 4
mu_0 = 200
# Complete


In [ ]:
# 🔴 Exercício 3: Funcao generica
def ttest_unilateral(dados, mu_0, lado='esquerda', alpha=0.05):
    # Implemente
    pass

## 🏆 Desafios

1. Compare a potência do teste T unilateral com a do teste T bicaudal para detectar um efeito de mesma magnitude.
2. Pesquise sobre **coeficiente de efeito (Cohen's d)** e calcule-o para um dos exemplos.
3. Explique em que situações um pesquisador escolheria o teste unilateral à esquerda em vez do bicaudal.

In [ ]:
# Desafio 2: Cohen's d (tamanho do efeito)
def cohens_d(dados, mu_0):
    d = (np.mean(dados) - mu_0) / np.std(dados, ddof=1)
    return d

d = cohens_d(lampadas, 1000)
print(f"Cohen's d para o exemplo das lâmpadas: {d:.4f}")
print(f"Interpretação: |d| ≈ {abs(d):.2f} → ", end="")
if abs(d) < 0.2:
    print("Efeito pequeno")
elif abs(d) < 0.5:
    print("Efeito médio")
else:
    print("Efeito grande")

## 📌 Resumo

- O **teste T unilateral à esquerda** verifica se μ < μ₀.
- H₀: μ ≥ μ₀, H₁: μ < μ₀. Região de rejeição na cauda esquerda.
- t = (x̄ - μ₀) / (s / √n), gl = n - 1.
- p-valor unilateral esquerda = P(T ≤ t) = t.cdf(t, gl).
- Para converter o p-valor bicaudal do ttest_1samp: divida por 2 se t < 0.
- Rejeitamos H₀ se t < -t_crit ou p-valor < α.

## 💡 Curiosidades

O teste T unilateral é mais poderoso que o bicaudal para detectar efeitos na direção especificada, mas só deve ser usado quando temos uma forte justificativa teórica. Na prática, muitos pesquisadores preferem usar o teste bicaudal e depois examinar o intervalo de confiança para determinar a direção do efeito — uma abordagem mais conservadora e informativa.

## ✅ Boas Práticas

1. Use o teste unilateral apenas com hipótese direcional pré-definida.
2. Verifique a normalidade dos dados (especialmente se n < 30).
3. Ao converter p-valor, verifique o sinal da estatística t.
4. Reporte também o intervalo de confiança (unilateral, se aplicável).
5. Considere o tamanho do efeito (Cohen's d) além do p-valor.

## ⚠️ Erros Comuns

| Erro | Como Evitar |
|------|-------------|
| Converter o p-valor para unilateral na direção errada | Verifique o sinal de t: se t > 0, a evidência é para μ > μ₀ |
| Usar teste unilateral depois de ver os dados | Isso infla o erro tipo I |
| Esquecer de verificar a normalidade | Use Shapiro-Wilk ou Q-Q plot para n pequeno |
| Confundir gl com n | gl = n - 1, não n |

## 📖 Referências

- [W3Statistics - T-Test Left Tailed](https://www.w3schools.com/statistics/statistics_ttest_left_tailed.php)
- [SciPy - ttest_1samp](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_1samp.html)
- [OpenIntro Statistics - Inference for Means](https://www.openintro.org/book/os/)

[◀ Anterior](S40_Teste_Proporcao_Bicaudal.ipynb) | [📖 Índice](S00_Index_Estatistica.ipynb) | [Próximo ▶](S42_Teste_Media_Bicaudal.ipynb)

---
🧠 **Quer uma abordagem mais intuitiva?**
Visite o [companheiro de analogia: Teste Unilateral a Esquerda (Media) — A Conta de Luz](S84_Teste_Media_Unilateral_E_Analogias.ipynb)
para aprender este conteudo com uma historia do dia a dia.
